ST554_Final Project   
Author: Joy Zhou  
Date: 4/21/2026

# Introduction   
This project is designed to assess the ability to use Apache Spark for processing streaming data and for training machine learning models in a scalable computing environment.


We will use a modified dataset from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city) on power consumption in Tetouan City, the project focuses on modeling the relationship between electricity consumption across different city zones and key influencing factors such as time of day, temperature, and humidity.   


By leveraging Spark's data processing and machine learning libraries, a predictive model will be developed using historical data and then applied to incoming data streams to perform real-time monitoring and prediction of power consumption. This project integrates concepts from big data processing, machine learning, and streaming analytics, highlighting the practical application of Spark in handling real-world, data-intensive problems.


# Part 1 Fitting the Model



The dataset power_ml_data.csv is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv.
The data should first be loaded into a pandas DataFrame using the `pd.read_csv()` function and then converted into a Spark DataFrame. In this analysis, `Power_Zone_3` is treated as the response variable, while all remaining variables are used as predictors.

## Read in data

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 10:41:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


- First we read `power_ml_data` into a standard pandas data frame named `power_data` using the pd.read_csv() function.


In [2]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


- Convert `power_data` to a spark data frame `power`

In [3]:
#Convert this to a spark data frame
power = spark.createDataFrame(power_data)
power.show(5)
#We are going to treat the Power_Zone_3 variable as our response variable

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Data transformation
We will fit an elastic net regression using cross-validation (CV), without performing an explicit training-test split. Instead, model selection and performance assessment are conducted entirely through cross-validation on the available dataset.   

Feature transformations are performed using Spark MLlib utilities, and the transformed variables are assembled into a machine learning pipeline to ensure a consistent and reproducible workflow.

- let's check the varaible types by inspectiong the dataset schema.


In [5]:
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



- Inspection of the schema shows that the Hour column is defined as a LongType. We therefore apply an SQLTransformer to cast this variable to DoubleType.

In [6]:
from pyspark.ml.feature import SQLTransformer

sqlTrans = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

To inspect the results of the SQL transformation, we apply sqlTrans to the power dataset and displayed a sample of the transformed records.

In [7]:
#inspect of records of sqlTrans
transformed_power = sqlTrans.transform(power)
transformed_power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335

- We apply a binarization step to the Hour variable using a cutoff of 6.5, creating a new binary feature, night_vs_day, to indicate nighttime (Hour < 6.5) versus daytime (Hour ≥ 6.5) for downstream modeling.
    

In [8]:
from pyspark.ml.feature import Binarizer

binaryTrans = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="night_vs_day")

#inspect the transformer results
binary = binaryTrans.transform(
    sqlTrans.transform(power))
binary.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0

- Although the Month variable is stored as a LongType, it represents a categorical feature rather than a continuous numeric value. Therefore, we apply one-hot encoding to the `Month` column to generate binary indicator variables, allowing the model to capture seasonal effects without assuming an ordinal or linear relationship among months.

In [9]:
from pyspark.ml.feature import OneHotEncoder, VectorAssembler

# Location one-hot encoding
month_encoder = OneHotEncoder(inputCol="Month", outputCol="Month_ind", dropLast=True)

We fit the month_encoder model to inspect the transformed records to ensure that the one-hot encoding is performed correctly.

In [10]:
#inspect the one-hot encode results
model = month_encoder.fit(binary)
encoded = model.transform(binary)
encoded.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|     Month_ind|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|       


- Principal Component Analysis (PCA) is used to reduce the dimensionality of the environmental variables Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows. The variables are assembled into a feature vector and standardized using [StandardScaler](https://www.sparkcodehub.com/pyspark/mllib/standard-scaler) to mitigate the influence of variables with larger magnitudes. PCA is then applied to extract two uncorrelated principal components, which provide lower‑dimensional representations for subsequent analysis and modeling.


In [13]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

#assemble raw features
assembler = VectorAssembler(inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol="pcaFeatures")

# Scale features (mean=0, std=1)
scaler = StandardScaler(inputCol="pcaFeatures", outputCol="scaledFeatures", withMean=True, withStd=True)

#PCA on scaled features
pca = PCA( k=2, inputCol="scaledFeatures", outputCol="pcaOutput")

#inspect the results
#set up pipeline
pipeline = Pipeline(stages=[assembler, scaler, pca])

# Fit and transform
pca_model = pipeline.fit(encoded)
pca_transformed = pca_model.transform(encoded)

pca_transformed.select("scaledFeatures", "pcaOutput").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------+---------------------------------------+
|scaledFeatures                                                                                      |pcaOutput                              |
+----------------------------------------------------------------------------------------------------+---------------------------------------+
|[-2.1079477438809433,0.3542085264292604,-0.7996341497278044,-0.6900839531060153,-0.6025312519793238]|[2.0690743213463714,0.8078678829472263]|
|[-2.1328903699941857,0.3991947174962608,-0.7996341497278044,-0.6900121009460847,-0.6028048802947571]|[2.1029210638806535,0.8152476222806394]|
|[-2.1502641992178924,0.3991947174962608,-0.800911098051248,-0.690042354487108,-0.6026841619203012]  |[2.112066263379101,0.8227151924829961] |
|[-2.183291676554047,0.4313277111155467,-0.7996341497278044,-0.6899326854008982,-0.6027163534868227] |[2.1435031847422192,0.8329135817940967]|

As the output above, we combined several environmental measurements into a single dataset and used PCA to compress them into two summary variables that capture most of the information. These summaries were then used instead of the original variables in later analyses.

- we will rename our response variable `Power_Zone_3` as label

In [14]:
sqlLabel = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

- Use `VectorAssembler()` to put all predictors into a single features vector, which includes:   
    - two fitted PCA features (pcaOutput)      
    - a binary Hour variable (night_vs_day)   
    - Power_Zone_1   
    - Power_Zone_2   
    - Month indicator variables (Month_ind)   
Since the PCA features were standardized in previous steps, the additional non‑PCA features will also be standardized to ensure consistency across all predictors.

In [15]:
#assemble assembleFeatures
features_assembler = VectorAssembler(
    inputCols=["pcaOutput", "night_vs_day", "Power_Zone_1", "Power_Zone_2", "Month_ind"],
    outputCol="assembledFeatures"
)

#scale again after adding non‑PCA features.
final_scaler = StandardScaler(
    inputCol="assembledFeatures",
    outputCol="features",
    withMean=True,
    withStd=True
)

assembled_df = features_assembler.transform(pca_transformed)
scaler_model = final_scaler.fit(assembled_df)
featurestrans = scaler_model.transform(assembled_df)

# Inspect assembled features
featurestrans.select("assembledFeatures", "features").show(5, truncate=False)

+------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|assembledFeatures                                                                   |features                                                                                                                                                                                                                                                                                                                              |
+------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------

From the output above, we confirmed that the transformation pipeline was completed and the the dataset is ready for modeling.

## Fit an Elastic Net Model
We next fit an Elastic Net regression model using the `CrossValidator()` and `LinearRegression()` function. Model hyperparameters are selected via cross-validation over a predefined grid consisting of all combinations of the following values:   

- regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

We build a modeling pipeline using the `Pipeline()` class from pyspark.ml, combining all required transformations and the target model into a single workflow.

In [16]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Elastic Net regression
lr = LinearRegression(featuresCol='features', labelCol='label') #create a liner regression instance

#define parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

- set up pipeline for modeling

In [17]:
pipeline = Pipeline(stages=[
    sqlTrans,              # Cast hour 
    binaryTrans,           # night_vs_day
    month_encoder,         # Month one-hot encoding
    sqlLabel,              # Rename response variable
    assembler,             # Assemble environmental vars to create pcaFeatures
    scaler,                # Scale pcaFeatures
    pca,                   # PCA produces pcaOutput
    features_assembler,    # Combine pcaOutput + indicators to get assembledFeatures
    final_scaler,          # Scale final feature vector to features
    lr                     # Elastic Net regression model
])

We apply `CrossValidator()` to carry out 5-fold cross-validation over the specified hyperparameter grid. Model performance is evaluated using `RegressionEvaluator()`, with RMSE specified via the metricName argument. This procedure will split the dataset into five folds. In each iteration, the model is trained on four folds and validated on the remaining fold for every hyperparameter combination. The performance metric (rmse) is then averaged across all five folds, and the model with the lowest average rmse is selected as the best model.

In [19]:
from pyspark.ml.evaluation import RegressionEvaluator
#initialize cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    ),
    numFolds=5,    #5-fold cross-validation
    seed = 42
)
#fitting the model
cv_model = cv.fit(power)

26/04/25 11:17:27 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/25 11:17:27 WARN Instrumentation: [0bfee83e] regParam is zero, which might cause numerical instability and overfitting.
26/04/25 11:17:28 WARN Instrumentation: [0bfee83e] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/25 11:17:31 WARN Instrumentation: [e5e4e8fe] regParam is zero, which might cause numerical instability and overfitting.
26/04/25 11:17:31 WARN Instrumentation: [e5e4e8fe] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/25 11:17:32 WARN Instrumentation: [97463977] regParam is zero, which might cause numerical instability and overfitting.
26/04/25 11:17:33 WARN Instrumentation: [97463977] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2

Evaluating the best model
- We will use the bestModel attribute to retrive the best model from cross validation
- Then, extract the final stage of the pipeline, the trained LinearRegression model, from the selected best model for further evaluation and interpretation.

In [20]:
# Retrieves the best pipeline model from the cross-validation
best_model = cv_model.bestModel
#Extracts the last stage of the pipeline
best_lr = best_model.stages[-1]  # LinearRegression is last stage

Report the optimal values of the tuning parameters (regParam and elasticNetParam) selected by cross‑validation using `getRegParam()` and `getElasticNetParam()` methods.

In [28]:
# Printing parameters for verification
print(f"Best regParam: {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

Best regParam: 0.05
Best elasticNetParam: 0.05


Report the CV error   
- The cross‑validation error is computed as the average RMSE across the five folds, evaluated over all hyperparameter combinations.

In [22]:
#report CV error
avg_rmse = min(cv_model.avgMetrics)
print("Best CV RMSE:", avg_rmse)

Best CV RMSE: 2174.9962489259447


Both parameters were selected as 0.05 because cross‑validation identified a model with mild, Ridge‑dominant regularization that best balances bias and variance for the standardized feature set.

Report training set RMSE
- The fitted regression model with the best parameters now has a transform method that can be used to make predictions using the entire dataset. The training set RMSE is obtained by applying the best fitted pipeline model to the entire dataset using the transform() method and evaluating prediction error using RMSE.

In [36]:
#apply the best fitted pipeline model to the entire dataset
train_predictions = best_model.transform(power)
# Evaluate RMSE
train_rmse = evaluator.evaluate(train_predictions)
print(f"Training RMSE: {train_rmse:.4f}")

Training RMSE: 2174.4490


The model outputs are used to generate predictions, from which a residual column is computed as the difference between the observed label and the predicted value. The `.withColumn()` method is used to create this residual. A resulting data frame is then displayed containing the residuals, the label column, and the model predictions.

In [27]:
from pyspark.sql.functions import col

residuals_df = train_predictions.withColumn("residual", col("label") - col("prediction"))

residuals_df.select( "residual", "label", "prediction").show(10, truncate=False)

+------------------+-----------+------------------+
|residual          |label      |prediction        |
+------------------+-----------+------------------+
|-695.2629810060898|20240.96386|20936.22684100609 |
|1429.4978719048668|20131.08434|18701.586468095134|
|1431.2442602265983|19668.43373|18237.189469773402|
|1283.3848937494076|18899.27711|17615.89221625059 |
|1425.0277973160955|18442.40964|17017.381842683906|
|1589.5061777943629|18130.12048|16540.61430220564 |
|1830.3760061055564|17945.06024|16114.684233894443|
|1717.238775797814 |17459.27711|15742.038334202185|
|1742.9841969908211|17025.54217|15282.55797300918 |
|1859.3175485773427|16794.21687|14934.899321422658|
+------------------+-----------+------------------+
only showing top 10 rows


## Conclusion
Using the Power dataset, an elastic net linear regression model was fitted through a pipeline and tuned via 5‑fold cross‑validation, with RMSE serving as the evaluation metric. The optimal regularization parameters were selected based on cross‑validated performance. The best‑fitted pipeline model was then applied to the entire dataset to compute the training set RMSE, which was 2174.449. Prediction residuals were subsequently calculated as the difference between the observed and predicted values.
During this process, transformations were applied to the hour and month variables, and all features were standardized prior to PCA and model fitting to mitigate the influence of variables with larger magnitudes. The use of a pipeline enabled efficient tracking of feature transformations and streamlined the modeling workflow, while Spark facilitated scalable and efficient model fitting. In the next section, methods for handling streaming data will be explored.

# Streaming Part
